In [21]:
# after we get a few gameweeks worth of data..

# below, we should put the most recent gameweek we have both an X and a y for. 
gameweek = 30

# the function will predict for that gameweek using previous gameweeks of data. We can then compare it to what was actually scored that week to see how we did. 

# after getting several weeks of data, we hope to see:

# - R^2 climbing to positive values, hopefully in the .2 to .3 range
# - Spearman climbing to above .2
# - Rank precision test getting 9/20 correct. If I am able to know half of the top 20 players each week, that would be really nice

from scipy.stats import spearmanr

def predict(gameweek: int):

    files = os.listdir(DATA_DIR)
    pattern = re.compile(r'^(X|y)_(\d+)\.csv$')

    file_map = {}
    for f in files:
        match = pattern.match(f)
        if match:
            kind, num = match.groups()
            file_map.setdefault(int(num), {})[kind] = f

    merged_dfs = []
    for gw, pair in file_map.items():
        if gw < gameweek and "X" in pair and "y" in pair:
            X = pd.read_csv(os.path.join(DATA_DIR, pair["X"]))
            y = pd.read_csv(os.path.join(DATA_DIR, pair["y"]))
            merged = X.merge(y, on="full_name", how="inner")
            merged = merged[merged["minutes_last_3"] >= 180]
            merged["gameweek"] = gw
            merged_dfs.append(merged)

    if not merged_dfs:
        raise ValueError("No training data found.")

    train_df = pd.concat(merged_dfs, ignore_index=True)
    print(f"Training on {len(train_df)} instances across {len(merged_dfs)} gameweeks")

    X_train = train_df.drop(columns=["gw_points", "gw_minutes", "full_name", "gameweek"])
    y_train = train_df["gw_points"]

    categorical = ["team_name", "player_position"]
    numeric = X_train.columns.difference(categorical)

    preprocessor = ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", "passthrough", numeric),
    ])

    model = Pipeline(steps=[
        ("prep", preprocessor),
        ("model", HistGradientBoostingRegressor(
            learning_rate=0.01,
            max_depth=4,
            max_iter=200,
            min_samples_leaf=20,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    if gameweek not in file_map or "X" not in file_map[gameweek]:
        raise ValueError(f"X_{gameweek}.csv not found")

    X_latest = pd.read_csv(os.path.join(DATA_DIR, file_map[gameweek]["X"]))
    X_latest = X_latest[X_latest["minutes_last_3"] >= 180]

    preds = model.predict(X_latest.drop(columns=["full_name", "gw_minutes"], errors="ignore"))

    pred_df = pd.DataFrame({
        "full_name":        X_latest["full_name"],
        "team_name":        X_latest["team_name"],
        "position":         X_latest["player_position"],
        "predicted_points": np.round(preds, 2),
    })

    if "y" in file_map.get(gameweek, {}):
        y_actual = pd.read_csv(os.path.join(DATA_DIR, file_map[gameweek]["y"]))
        pred_df = pred_df.merge(y_actual[["full_name", "gw_points"]], on="full_name", how="left")
        pred_df.rename(columns={"gw_points": "actual_points"}, inplace=True)

        scored = pred_df.dropna(subset=["actual_points"])
        mae = mean_absolute_error(scored["actual_points"], scored["predicted_points"])
        r2  = r2_score(scored["actual_points"], scored["predicted_points"])

        print(f"\nGW{gameweek} Evaluation ({len(scored)} players with actuals)")
        print(f"  MAE : {mae:.2f}")
        print(f"  R²  : {r2:.3f}")

    print(f"\nTop 10 predicted:\n{pred_df.sort_values('predicted_points', ascending=False).head(10).to_string(index=False)}")

    return pred_df




    
pred_df = predict(gameweek)





scored = pred_df.dropna(subset=["actual_points"])
mae = mean_absolute_error(scored["actual_points"], scored["predicted_points"])
r2  = r2_score(scored["actual_points"], scored["predicted_points"])
rho, p_value = spearmanr(scored["actual_points"], scored["predicted_points"])

print(f"\nGW{gameweek} Evaluation ({len(scored)} players with actuals)")
print(f"  MAE     : {mae:.2f}")
print(f"  R²      : {r2:.3f}")
print(f"  Spearman: {rho:.3f}  (p={p_value:.3f})")

def top_n_precision(df, n=20):
    top_predicted = set(df.nlargest(n, "predicted_points")["full_name"])
    top_actual    = set(df.nlargest(n, "actual_points")["full_name"])
    overlap = len(top_predicted & top_actual)
    print(f"  Top-{n} Precision: {overlap}/{n} players correctly identified")

top_n_precision(scored, n=20)

Training on 167 instances across 1 gameweeks

GW30 Evaluation (95 players with actuals)
  MAE : 2.64
  R²  : -0.273

Top 10 predicted:
                   full_name      team_name   position  predicted_points  actual_points
      Bruno Borges Fernandes        Man Utd Midfielder              7.62             10
         Marcos Senesi Barón    Bournemouth   Defender              6.55             10
             Antoine Semenyo       Man City Midfielder              5.98              2
             James Tarkowski        Everton   Defender              5.77              0
              Jurriën Timber        Arsenal   Defender              5.66              1
Gabriel dos Santos Magalhães        Arsenal   Defender              5.66              9
             Adrien Truffert    Bournemouth   Defender              5.02             11
                James Garner        Everton Midfielder              4.74              4
            Mikkel Damsgaard      Brentford Midfielder              4.67 

In [ ]:
gameweek = 30
pred_df = predict(gameweek)

scored = pred_df.dropna(subset=["actual_points"])
mae = mean_absolute_error(scored["actual_points"], scored["predicted_points"])
r2  = r2_score(scored["actual_points"], scored["predicted_points"])
rho, p_value = spearmanr(scored["actual_points"], scored["predicted_points"])

print(f"\nGW{gameweek} Evaluation ({len(scored)} players with actuals)")
print(f"  MAE     : {mae:.2f}")
print(f"  R²      : {r2:.3f}")
print(f"  Spearman: {rho:.3f}  (p={p_value:.3f})")

def top_n_precision(df, n=20):
    top_predicted = set(df.nlargest(n, "predicted_points")["full_name"])
    top_actual    = set(df.nlargest(n, "actual_points")["full_name"])
    overlap = len(top_predicted & top_actual)
    print(f"  Top-{n} Precision: {overlap}/{n} players correctly identified")

top_n_precision(scored, n=20)